In [6]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer

In [7]:
# Load dataset
df = pd.read_csv("data/data.csv")
df.head()

,R_fighter,B_fighter,date,R_height,R_reach,R_stance,R_age,B_height,B_reach,B_stance,...,B_win_by_KO/TKO,B_win_by_SUB,winner,win_type,finish_type,decision_type,last_round,format,title_bout,weight_class
0,Tito Ortiz,Elvis Sinosic,2001-06-29,190.50,74.0,Orthodox,26,190.50,77.0,Orthodox,...,0.0,1.0,red,finish,KO/TKO,NaN,1,5,True,Light Heavyweight
1,Tito Ortiz,Vladimir Matyushenko,2001-09-28,190.50,74.0,Orthodox,26,182.88,74.0,Orthodox,...,0.0,0.0,red,decision,NaN,unanimous,5,5,True,Light Heavyweight
2,BJ Penn,Caol Uno,2001-11-02,175.26,70.0,Orthodox,22,170.18,70.0,Southpaw,...,0.5,0.0,red,finish,KO/TKO,NaN,1,3,False,Lightweight
3,Jens Pulver,BJ Penn,2002-01-11,170.18,70.0,Southpaw,27,175.26,70.0,Orthodox,...,1.0,0.0,red,decision,NaN,majority,5,5,True,Lightweight
4,Evan Tanner,Elvis Sinosic,2002-03-22,182.88,74.0,Orthodox,31,190.50,77.0,Orthodox,...,0.0,0.5,red,finish,KO/TKO,NaN,1,3,False,Light Heavyweight


## Preprocessing

In [8]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort fights so earlier fights come first
df = df.sort_values("date").reset_index(drop=True)

# Convert title_bout to numeric (True/False → 1/0)
df["title_bout"] = df["title_bout"].astype(int)

In [9]:
# Separate Target and Features
# Target label
y1 = df["winner"]
# y = df["winner", "win_type", "finish_type", "decision_type", "last_round"] #all targets

# Drop columns that won't be used as features
X = df.drop(columns=[
    "winner",
    "win_type",
    "finish_type",
    "decision_type",
    "last_round",
    "R_fighter",
    "B_fighter",
    "date"
])

X.shape

(5885, 123)

In [10]:
# Fighter-level KNN imputation (one fighter per row)
# Avoids using opponent-side features directly to fill a fighter's missing stats.

# Build paired fighter feature suffixes available on both corners
paired_suffixes = sorted(
    {
        col[2:]
        for col in X.columns
        if col.startswith("R_") and f"B_{col[2:]}" in X.columns
    }
)

# Keep only numeric paired fighter features
fighter_num_suffixes = [
    s
    for s in paired_suffixes
    if s != "stance"
    and pd.api.types.is_numeric_dtype(X[f"R_{s}"])
    and pd.api.types.is_numeric_dtype(X[f"B_{s}"])
]

# Shared numeric context columns can help define fighter similarity
shared_numeric_cols = [
    col
    for col in ["format", "title_bout"]
    if col in X.columns and pd.api.types.is_numeric_dtype(X[col])
]

row_ids = np.arange(len(X))

red_fighters = pd.DataFrame({"_row_id": row_ids, "_corner": "R"})
blue_fighters = pd.DataFrame({"_row_id": row_ids, "_corner": "B"})

for s in fighter_num_suffixes:
    red_fighters[s] = X[f"R_{s}"].values
    blue_fighters[s] = X[f"B_{s}"].values

for col in shared_numeric_cols:
    red_fighters[col] = X[col].values
    blue_fighters[col] = X[col].values

fighters_long = pd.concat([red_fighters, blue_fighters], ignore_index=True)

# KNN in fighter space
imputer = KNNImputer(n_neighbors=5)
impute_cols = fighter_num_suffixes + shared_numeric_cols
fighters_long[impute_cols] = imputer.fit_transform(fighters_long[impute_cols])

# Write imputed fighter features back to matchup rows
red_imputed = fighters_long[fighters_long["_corner"] == "R"].set_index("_row_id")
blue_imputed = fighters_long[fighters_long["_corner"] == "B"].set_index("_row_id")

for s in fighter_num_suffixes:
    X[f"R_{s}"] = red_imputed.loc[row_ids, s].values
    X[f"B_{s}"] = blue_imputed.loc[row_ids, s].values

X.shape

(5885, 123)

In [11]:
# Categorical columns
cat_cols = ["R_stance", "B_stance", "weight_class"]

# Numeric columns
num_cols = X.columns.drop(cat_cols)

# Keep numeric dtypes consistent for imputation/scaling
X[num_cols] = X[num_cols].astype(float)

# Scale numeric features and encode all categorical columns
transformer = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)

# Fit & transform the features
X_transformed = transformer.fit_transform(X)

X_transformed.shape

(5885, 141)

## Train/Test Split

In [12]:
import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import TensorDataset, DataLoader

# label encoding converts strings -> integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y1.values.ravel())
num_classes = len(label_encoder.classes_)

# 90/10 split (temporal since chronological, not random stratified)
split_index = int(len(X_transformed) * 0.9)
X_train = X_transformed[:split_index]
X_test = X_transformed[split_index:]
y_train = y_encoded[:split_index]
y_test = y_encoded[split_index:]

# Temporal split check (date range)
print("\nTrain date range:", df["date"].iloc[:split_index].min(), "to", df["date"].iloc[:split_index].max())
print("Test date range:", df["date"].iloc[split_index:].min(), "to", df["date"].iloc[split_index:].max())


# Converting to tensors
X_train_tensor = torch.tensor(X_train).float()
X_test_tensor = torch.tensor(X_test).float()
y_train_tensor = torch.tensor(y_train.astype("int64"), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.astype("int64"), dtype=torch.long)

# DataLoaders
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print("\nX train tensor shape: ",  X_train_tensor.shape)
print("X test tensor shape: ",  X_test_tensor.shape)
print("num classes: ", num_classes)



Train date range: 2001-06-29 00:00:00 to 2024-09-28 00:00:00
Test date range: 2024-09-28 00:00:00 to 2026-02-28 00:00:00

X train tensor shape:  torch.Size([5296, 141])
X test tensor shape:  torch.Size([589, 141])
num classes:  3


## Baseline Models (with draws)

### Majority and Random Baselines

In [13]:
from sklearn.metrics import accuracy_score, f1_score

# Majority baseline: always predict the most frequent class from train set
majority_class = np.bincount(y_train).argmax()
y_pred_majority = np.full_like(y_test, fill_value=majority_class)

# Random baseline: sample labels using empirical class probabilities from train set
rng = np.random.default_rng(42)
class_probs = np.bincount(y_train) / len(y_train)
y_pred_random = rng.choice(np.arange(num_classes), size=len(y_test), p=class_probs)

print("Majority class:", label_encoder.inverse_transform([majority_class])[0])
print("\nMajority baseline")
print("Accuracy:", round(accuracy_score(y_test, y_pred_majority), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_majority, average="macro"), 4))

print("\nRandom baseline (train-prior sampling)")
print("Accuracy:", round(accuracy_score(y_test, y_pred_random), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_random, average="macro"), 4))

Majority class: red

Majority baseline
Accuracy: 0.5535
Macro F1: 0.2375

Random baseline (train-prior sampling)
Accuracy: 0.511
Macro F1: 0.3333


The majority baseline shows that predicting red gets decent accuracy because red is the most common class, but it performs very poorly across classes overall. The random baseline has lower accuracy but better class balance, which is why its macro F1 is higher.

### Logistic Regression

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

log_reg = LogisticRegression(
    solver="saga",
    max_iter=2000,
    random_state=42,
)

log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)

print("Logistic Regression baseline")
print("Accuracy:", round(accuracy_score(y_test, y_pred_lr), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_lr, average="macro"), 4))
print("Log Loss:", round(log_loss(y_test, y_proba_lr), 4))

Logistic Regression baseline
Accuracy: 0.6452
Macro F1: 0.4237
Log Loss: 0.6764


/Users/Ryan/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


The logistic regression baseline beats both the majority and random baselines by a decent margin. The model is learning meaningful signal from the features, not just class frequency, and it's handling class imbalance better.

### MLP

In [15]:
# Training & Evaluating a deeper MLP baseline with early stopping
from sklearn.metrics import accuracy_score, f1_score
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Deeper but still lightweight baseline network
model = torch.nn.Sequential(
    torch.nn.Linear(X_train_tensor.shape[1], 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 64),
    torch.nn.ReLU(),
    torch.nn.Linear(64, num_classes),
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)


def eval_loader(loader):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, macro_f1


num_epochs = 80
patience = 8
min_delta = 1e-4
best_test_loss = float("inf")
best_state = None
wait = 0

for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss_sum = 0.0

    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)

    train_loss = train_loss_sum / len(train_loader.dataset)
    test_loss, test_acc, test_f1 = eval_loader(test_loader)

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | test_loss={test_loss:.4f}"
        )

    if test_loss < best_test_loss - min_delta:
        best_test_loss = test_loss
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch} | best_test_loss={best_test_loss:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state)

train_loss, train_acc, train_f1 = eval_loader(train_loader)
test_loss, test_acc, test_f1 = eval_loader(test_loader)

print("Deeper MLP baseline results")
print(f"Train Accuracy: {train_acc:.4f} | Train Macro F1: {train_f1:.4f}")
print(f"Test  Accuracy: {test_acc:.4f} | Test  Macro F1: {test_f1:.4f}")
print(f"Test  Loss: {test_loss:.4f}")

/Users/Ryan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/Ryan/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 01/80 | train_loss=0.8461 | test_loss=0.7239
Epoch 05/80 | train_loss=0.6296 | test_loss=0.6685
Epoch 10/80 | train_loss=0.5727 | test_loss=0.6922
Early stopping at epoch 14 | best_test_loss=0.6629
Deeper MLP baseline results
Train Accuracy: 0.6954 | Train Macro F1: 0.4416
Test  Accuracy: 0.6587 | Test  Macro F1: 0.4332
Test  Loss: 0.6629


## No-Draw Preprocessing + Same 4 Baselines

In [16]:
# 1) remove draw outcomes, 2) drop draw feature columns, 3) rerun same 4 baselines

import copy
import numpy as np
import pandas as pd
import torch

from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------
# Preprocessing (no-draw variant)
# -----------------------------
df_nd = pd.read_csv("data/data.csv")
df_nd["date"] = pd.to_datetime(df_nd["date"])

df_nd = df_nd[df_nd["winner"] != "draw"].copy()
df_nd = df_nd.sort_values("date").reset_index(drop=True)
df_nd["title_bout"] = df_nd["title_bout"].astype(int)

y_nd = df_nd["winner"]

X_nd = df_nd.drop(columns=[
    "winner",
    "win_type",
    "finish_type",
    "decision_type",
    "last_round",
    "R_fighter",
    "B_fighter",
    "date",
    "R_draws",
    "B_draws",
], errors="ignore")

# Fighter-level KNN imputation (same approach as original)
paired_suffixes_nd = sorted(
    {
        col[2:]
        for col in X_nd.columns
        if col.startswith("R_") and f"B_{col[2:]}" in X_nd.columns
    }
)

fighter_num_suffixes_nd = [
    s
    for s in paired_suffixes_nd
    if s != "stance"
    and pd.api.types.is_numeric_dtype(X_nd[f"R_{s}"])
    and pd.api.types.is_numeric_dtype(X_nd[f"B_{s}"])
]

shared_numeric_cols_nd = [
    col
    for col in ["format", "title_bout"]
    if col in X_nd.columns and pd.api.types.is_numeric_dtype(X_nd[col])
]

row_ids_nd = np.arange(len(X_nd))
red_fighters_nd = pd.DataFrame({"_row_id": row_ids_nd, "_corner": "R"})
blue_fighters_nd = pd.DataFrame({"_row_id": row_ids_nd, "_corner": "B"})

for s in fighter_num_suffixes_nd:
    red_fighters_nd[s] = X_nd[f"R_{s}"].values
    blue_fighters_nd[s] = X_nd[f"B_{s}"].values

for col in shared_numeric_cols_nd:
    red_fighters_nd[col] = X_nd[col].values
    blue_fighters_nd[col] = X_nd[col].values

fighters_long_nd = pd.concat([red_fighters_nd, blue_fighters_nd], ignore_index=True)

imputer_nd = KNNImputer(n_neighbors=5)
impute_cols_nd = fighter_num_suffixes_nd + shared_numeric_cols_nd
fighters_long_nd[impute_cols_nd] = imputer_nd.fit_transform(fighters_long_nd[impute_cols_nd])

red_imputed_nd = fighters_long_nd[fighters_long_nd["_corner"] == "R"].set_index("_row_id")
blue_imputed_nd = fighters_long_nd[fighters_long_nd["_corner"] == "B"].set_index("_row_id")

for s in fighter_num_suffixes_nd:
    X_nd[f"R_{s}"] = red_imputed_nd.loc[row_ids_nd, s].values
    X_nd[f"B_{s}"] = blue_imputed_nd.loc[row_ids_nd, s].values

cat_cols_nd = ["R_stance", "B_stance", "weight_class"]
num_cols_nd = X_nd.columns.drop(cat_cols_nd)
X_nd[num_cols_nd] = X_nd[num_cols_nd].astype(float)

transformer_nd = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols_nd),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_nd),
    ],
    remainder="drop",
)

X_transformed_nd = transformer_nd.fit_transform(X_nd)

# -----------------------------
# Temporal split + tensors
# -----------------------------
label_encoder_nd = LabelEncoder()
y_encoded_nd = label_encoder_nd.fit_transform(y_nd.values.ravel())
num_classes_nd = len(label_encoder_nd.classes_)

split_index_nd = int(len(X_transformed_nd) * 0.9)
X_train_nd = X_transformed_nd[:split_index_nd]
X_test_nd = X_transformed_nd[split_index_nd:]
y_train_nd = y_encoded_nd[:split_index_nd]
y_test_nd = y_encoded_nd[split_index_nd:]

X_train_tensor_nd = torch.tensor(X_train_nd).float()
X_test_tensor_nd = torch.tensor(X_test_nd).float()
y_train_tensor_nd = torch.tensor(y_train_nd.astype("int64"), dtype=torch.long)
y_test_tensor_nd = torch.tensor(y_test_nd.astype("int64"), dtype=torch.long)

train_ds_nd = TensorDataset(X_train_tensor_nd, y_train_tensor_nd)
test_ds_nd = TensorDataset(X_test_tensor_nd, y_test_tensor_nd)
train_loader_nd = DataLoader(train_ds_nd, batch_size=64, shuffle=True)
test_loader_nd = DataLoader(test_ds_nd, batch_size=64, shuffle=False)

print("No-draw rows:", len(df_nd))
print("Feature matrix shape (no-draw):", X_transformed_nd.shape)
print("Classes (no-draw):", list(label_encoder_nd.classes_))

# Joint labels for simultaneous winner + win_type prediction
# Example class: "red__decision"
y_joint_all = (df_nd["winner"].astype(str) + "__" + df_nd["win_type"].astype(str)).values
y_joint_train = y_joint_all[:split_index_nd]
y_joint_test = y_joint_all[split_index_nd:]

joint_encoder = LabelEncoder()
y_joint_train_enc = joint_encoder.fit_transform(y_joint_train)
y_joint_test_enc = joint_encoder.transform(y_joint_test)
num_joint_classes = len(joint_encoder.classes_)


def split_joint_label(joint_labels):
    winners = []
    win_types = []
    for label in joint_labels:
        w, wt = label.split("__", 1)
        winners.append(w)
        win_types.append(wt)
    return np.array(winners), np.array(win_types)


def report_joint_metrics(name, y_true_joint, y_pred_joint):
    true_w, true_t = split_joint_label(y_true_joint)
    pred_w, pred_t = split_joint_label(y_pred_joint)

    print(f"\n[JOINT] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("joint_acc:", round(accuracy_score(y_true_joint, y_pred_joint), 4))
    print("joint_macro_f1:", round(f1_score(y_true_joint, y_pred_joint, average="macro"), 4))


def report_hier_metrics(name, true_w, true_t, pred_w, pred_t):
    joint_acc = np.mean((pred_w == true_w) & (pred_t == true_t))
    print(f"\n[HIERARCHICAL] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("joint_acc:", round(float(joint_acc), 4))


def compare_joint_vs_hier(name, y_true_joint, y_pred_joint, pred_w_h, pred_t_h):
    true_w, true_t = split_joint_label(y_true_joint)
    joint_acc_joint = accuracy_score(y_true_joint, y_pred_joint)
    joint_acc_hier = np.mean((pred_w_h == true_w) & (pred_t_h == true_t))

    print(f"\n[COMPARISON] {name} | Joint vs Hierarchical")
    print("joint_model_joint_acc:", round(joint_acc_joint, 4))
    print("hier_model_joint_acc:", round(float(joint_acc_hier), 4))
    print("delta_hier_minus_joint:", round(float(joint_acc_hier - joint_acc_joint), 4))


# Baseline execution is split across separate cells below:
# 1) winner-only baselines
# 2) joint baselines
# 3) hierarchical baselines


def train_simple_mlp(train_loader, test_loader, input_dim, output_dim, device, epochs=80):
    model = torch.nn.Sequential(
        torch.nn.Linear(input_dim, 128),
        torch.nn.ReLU(),
        torch.nn.Linear(128, 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, output_dim),
    ).to(device)

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)

    def eval_loader(loader):
        model.eval()
        total_loss = 0.0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for features, labels in loader:
                features, labels = features.to(device), labels.to(device)
                logits = model(features)
                loss = criterion(logits, labels)
                total_loss += loss.item() * labels.size(0)
                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())
        avg_loss = total_loss / len(loader.dataset)
        return avg_loss, np.array(all_preds), np.array(all_labels)

    best_state = None
    best_test_loss = float("inf")
    wait = 0

    for _ in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        test_loss, _, _ = eval_loader(test_loader)
        if test_loss < best_test_loss - 1e-4:
            best_test_loss = test_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= 8:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, eval_loader


print("No-draw rows:", len(df_nd))
print("Feature matrix shape (no-draw):", X_transformed_nd.shape)
print("Classes (no-draw):", list(label_encoder_nd.classes_))

No-draw rows: 5844
Feature matrix shape (no-draw): (5844, 139)
Classes (no-draw): ['blue', 'red']
No-draw rows: 5844
Feature matrix shape (no-draw): (5844, 139)
Classes (no-draw): ['blue', 'red']


### No-Draw Baselines: Winner-Only

In [17]:
# Winner-only baselines on no-draw data
rng_nd = np.random.default_rng(42)
device_nd = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Majority
majority_class_nd = np.bincount(y_train_nd).argmax()
y_pred_majority_nd = np.full_like(y_test_nd, fill_value=majority_class_nd)
print("\n[WINNER-ONLY] Majority baseline")
print("majority_class:", label_encoder_nd.inverse_transform([majority_class_nd])[0])
print("accuracy:", round(accuracy_score(y_test_nd, y_pred_majority_nd), 4))
print("macro_f1:", round(f1_score(y_test_nd, y_pred_majority_nd, average="macro"), 4))

# Random prior
class_probs_nd = np.bincount(y_train_nd) / len(y_train_nd)
y_pred_random_nd = rng_nd.choice(np.arange(num_classes_nd), size=len(y_test_nd), p=class_probs_nd)
print("\n[WINNER-ONLY] Random baseline")
print("accuracy:", round(accuracy_score(y_test_nd, y_pred_random_nd), 4))
print("macro_f1:", round(f1_score(y_test_nd, y_pred_random_nd, average="macro"), 4))

# Logistic
log_reg_nd = LogisticRegression(solver="saga", max_iter=2000, random_state=42)
log_reg_nd.fit(X_train_nd, y_train_nd)
y_pred_lr_nd = log_reg_nd.predict(X_test_nd)
y_proba_lr_nd = log_reg_nd.predict_proba(X_test_nd)
print("\n[WINNER-ONLY] Logistic baseline")
print("accuracy:", round(accuracy_score(y_test_nd, y_pred_lr_nd), 4))
print("macro_f1:", round(f1_score(y_test_nd, y_pred_lr_nd, average="macro"), 4))
print("log_loss:", round(log_loss(y_test_nd, y_proba_lr_nd), 4))

# MLP (shared helper)
model_nd, eval_winner_mlp_nd = train_simple_mlp(
    train_loader_nd,
    test_loader_nd,
    input_dim=X_train_tensor_nd.shape[1],
    output_dim=num_classes_nd,
    device=device_nd,
    epochs=80,
)

test_loss_nd, y_pred_mlp_nd, y_true_mlp_nd = eval_winner_mlp_nd(test_loader_nd)
test_acc_nd = accuracy_score(y_true_mlp_nd, y_pred_mlp_nd)
test_f1_nd = f1_score(y_true_mlp_nd, y_pred_mlp_nd, average="macro")
print("\n[WINNER-ONLY] MLP baseline")
print("accuracy:", round(test_acc_nd, 4))
print("macro_f1:", round(test_f1_nd, 4))
print("loss:", round(test_loss_nd, 4))


[WINNER-ONLY] Majority baseline
majority_class: red
accuracy: 0.5556
macro_f1: 0.3571

[WINNER-ONLY] Random baseline
accuracy: 0.5282
macro_f1: 0.5149

[WINNER-ONLY] Logistic baseline
accuracy: 0.6479
macro_f1: 0.6357
log_loss: 0.635

[WINNER-ONLY] MLP baseline
accuracy: 0.6632
macro_f1: 0.6584
loss: 0.6327


### No-Draw Baselines: Joint (Winner + Win Type)

In [18]:
# Joint baselines on no-draw data
y_pred_joint_map = {}

# Majority
majority_joint_class = pd.Series(y_joint_train).value_counts().idxmax()
y_pred_majority_joint = np.full_like(y_joint_test, fill_value=majority_joint_class)
y_pred_joint_map["majority"] = y_pred_majority_joint
report_joint_metrics("Majority baseline", y_joint_test, y_pred_majority_joint)

# Random prior
joint_classes, joint_counts = np.unique(y_joint_train, return_counts=True)
joint_probs = joint_counts / joint_counts.sum()
y_pred_random_joint = rng_nd.choice(joint_classes, size=len(y_joint_test), p=joint_probs)
y_pred_joint_map["random"] = y_pred_random_joint
report_joint_metrics("Random baseline", y_joint_test, y_pred_random_joint)

# Logistic
log_reg_joint_nd = LogisticRegression(solver="saga", max_iter=3000, random_state=42)
log_reg_joint_nd.fit(X_train_nd, y_joint_train_enc)
y_pred_lr_joint_enc = log_reg_joint_nd.predict(X_test_nd)
y_pred_lr_joint = joint_encoder.inverse_transform(y_pred_lr_joint_enc)
y_pred_joint_map["logistic"] = y_pred_lr_joint
report_joint_metrics("Logistic baseline", y_joint_test, y_pred_lr_joint)

# MLP (shared helper)
y_joint_train_tensor_nd = torch.tensor(y_joint_train_enc.astype("int64"), dtype=torch.long)
y_joint_test_tensor_nd = torch.tensor(y_joint_test_enc.astype("int64"), dtype=torch.long)
train_ds_joint_nd = TensorDataset(X_train_tensor_nd, y_joint_train_tensor_nd)
test_ds_joint_nd = TensorDataset(X_test_tensor_nd, y_joint_test_tensor_nd)
train_loader_joint_nd = DataLoader(train_ds_joint_nd, batch_size=64, shuffle=True)
test_loader_joint_nd = DataLoader(test_ds_joint_nd, batch_size=64, shuffle=False)

model_joint_nd, eval_joint_mlp_nd = train_simple_mlp(
    train_loader_joint_nd,
    test_loader_joint_nd,
    input_dim=X_train_tensor_nd.shape[1],
    output_dim=num_joint_classes,
    device=device_nd,
    epochs=80,
)

_, y_pred_joint_mlp_enc, _ = eval_joint_mlp_nd(test_loader_joint_nd)
y_pred_joint_mlp = joint_encoder.inverse_transform(y_pred_joint_mlp_enc)
y_pred_joint_map["mlp"] = y_pred_joint_mlp
report_joint_metrics("MLP baseline", y_joint_test, y_pred_joint_mlp)


[JOINT] Majority baseline
winner_acc: 0.5556
win_type_acc: 0.4718
joint_acc: 0.265


joint_macro_f1: 0.1047

[JOINT] Random baseline
winner_acc: 0.5162
win_type_acc: 0.4667
joint_acc: 0.2359
joint_macro_f1: 0.2325

[JOINT] Logistic baseline
winner_acc: 0.6376
win_type_acc: 0.6017
joint_acc: 0.4103
joint_macro_f1: 0.404

[JOINT] MLP baseline
winner_acc: 0.6598
win_type_acc: 0.5932
joint_acc: 0.4171
joint_macro_f1: 0.4102


### No-Draw Baselines: Hierarchical (Winner -> Win Type)

In [19]:
# Hierarchical baselines on no-draw data
from scipy import sparse as sp

true_w_train, true_t_train = split_joint_label(y_joint_train)
true_w_test, true_t_test = split_joint_label(y_joint_test)

# Shared win_type labels for hierarchical stage-2 models
y_win_type_train = df_nd["win_type"].iloc[:split_index_nd].values
y_win_type_test = df_nd["win_type"].iloc[split_index_nd:].values
win_type_encoder = LabelEncoder()
y_win_type_train_enc = win_type_encoder.fit_transform(y_win_type_train)

# 1) Majority
majority_winner = pd.Series(true_w_train).value_counts().idxmax()
majority_type_by_winner = {
    w: pd.Series(true_t_train[true_w_train == w]).value_counts().idxmax()
    for w in np.unique(true_w_train)
}
pred_w_h_majority = np.full_like(true_w_test, fill_value=majority_winner)
pred_t_h_majority = np.array([majority_type_by_winner[w] for w in pred_w_h_majority])
report_hier_metrics("Majority baseline", true_w_test, true_t_test, pred_w_h_majority, pred_t_h_majority)
compare_joint_vs_hier("Majority baseline", y_joint_test, y_pred_joint_map["majority"], pred_w_h_majority, pred_t_h_majority)

# 2) Random
winner_classes, winner_counts = np.unique(true_w_train, return_counts=True)
winner_probs = winner_counts / winner_counts.sum()
type_classes_by_winner, type_probs_by_winner = {}, {}
for w in winner_classes:
    t_vals, t_counts = np.unique(true_t_train[true_w_train == w], return_counts=True)
    type_classes_by_winner[w] = t_vals
    type_probs_by_winner[w] = t_counts / t_counts.sum()

pred_w_h_random = rng_nd.choice(winner_classes, size=len(true_w_test), p=winner_probs)
pred_t_h_random = np.array([
    rng_nd.choice(type_classes_by_winner[w], p=type_probs_by_winner[w])
    for w in pred_w_h_random
])
report_hier_metrics("Random baseline", true_w_test, true_t_test, pred_w_h_random, pred_t_h_random)
compare_joint_vs_hier("Random baseline", y_joint_test, y_pred_joint_map["random"], pred_w_h_random, pred_t_h_random)

# 3) Logistic
pred_w_h_lr = label_encoder_nd.inverse_transform(y_pred_lr_nd)
winner_train_lr_enc = sp.csr_matrix(y_train_nd.reshape(-1, 1))
winner_test_lr_enc = sp.csr_matrix(y_pred_lr_nd.reshape(-1, 1))
X_train_h_lr = sp.hstack([X_train_nd, winner_train_lr_enc], format="csr")
X_test_h_lr = sp.hstack([X_test_nd, winner_test_lr_enc], format="csr")

log_reg_win_type_nd = LogisticRegression(solver="saga", max_iter=3000, random_state=42)
log_reg_win_type_nd.fit(X_train_h_lr, y_win_type_train_enc)
y_pred_win_type_h_lr = win_type_encoder.inverse_transform(log_reg_win_type_nd.predict(X_test_h_lr))
report_hier_metrics("Logistic baseline", true_w_test, true_t_test, pred_w_h_lr, y_pred_win_type_h_lr)
compare_joint_vs_hier("Logistic baseline", y_joint_test, y_pred_joint_map["logistic"], pred_w_h_lr, y_pred_win_type_h_lr)

# 4) MLP
with torch.no_grad():
    winner_logits_h_mlp = model_nd(X_test_tensor_nd.to(device_nd))
    winner_pred_h_mlp_enc = np.asarray(torch.argmax(winner_logits_h_mlp, dim=1).cpu().tolist(), dtype=np.int64)

pred_w_h_mlp = label_encoder_nd.inverse_transform(winner_pred_h_mlp_enc)
X_train_nd_dense = X_train_nd.toarray() if hasattr(X_train_nd, "toarray") else np.asarray(X_train_nd)
X_test_nd_dense = X_test_nd.toarray() if hasattr(X_test_nd, "toarray") else np.asarray(X_test_nd)

X_train_h_mlp = np.hstack([X_train_nd_dense, y_train_nd.reshape(-1, 1)]).astype(np.float32)
X_test_h_mlp = np.hstack([X_test_nd_dense, winner_pred_h_mlp_enc.reshape(-1, 1)]).astype(np.float32)

y_train_h_mlp_t = torch.tensor(y_win_type_train_enc.astype("int64"), dtype=torch.long)
train_ds_h_mlp = TensorDataset(torch.tensor(X_train_h_mlp, dtype=torch.float32), y_train_h_mlp_t)
test_ds_h_mlp = TensorDataset(torch.tensor(X_test_h_mlp, dtype=torch.float32), torch.zeros(len(X_test_h_mlp), dtype=torch.long))
train_loader_h_mlp = DataLoader(train_ds_h_mlp, batch_size=64, shuffle=True)
test_loader_h_mlp = DataLoader(test_ds_h_mlp, batch_size=64, shuffle=False)

model_h_mlp, eval_h_mlp = train_simple_mlp(
    train_loader_h_mlp,
    test_loader_h_mlp,
    input_dim=X_train_h_mlp.shape[1],
    output_dim=len(win_type_encoder.classes_),
    device=device_nd,
    epochs=20,
)

_, pred_t_h_mlp_enc, _ = eval_h_mlp(test_loader_h_mlp)
y_pred_win_type_h_mlp = win_type_encoder.inverse_transform(pred_t_h_mlp_enc)
report_hier_metrics("MLP baseline", true_w_test, true_t_test, pred_w_h_mlp, y_pred_win_type_h_mlp)
compare_joint_vs_hier("MLP baseline", y_joint_test, y_pred_joint_map["mlp"], pred_w_h_mlp, y_pred_win_type_h_mlp)


[HIERARCHICAL] Majority baseline
winner_acc: 0.5556
win_type_acc: 0.4718
joint_acc: 0.265

[COMPARISON] Majority baseline | Joint vs Hierarchical
joint_model_joint_acc: 0.265
hier_model_joint_acc: 0.265
delta_hier_minus_joint: 0.0



[HIERARCHICAL] Random baseline
winner_acc: 0.535
win_type_acc: 0.5214
joint_acc: 0.2786

[COMPARISON] Random baseline | Joint vs Hierarchical
joint_model_joint_acc: 0.2359
hier_model_joint_acc: 0.2786
delta_hier_minus_joint: 0.0427

[HIERARCHICAL] Logistic baseline
winner_acc: 0.6479
win_type_acc: 0.6034
joint_acc: 0.4068

[COMPARISON] Logistic baseline | Joint vs Hierarchical
joint_model_joint_acc: 0.4103
hier_model_joint_acc: 0.4068
delta_hier_minus_joint: -0.0034

[HIERARCHICAL] MLP baseline
winner_acc: 0.6632
win_type_acc: 0.588
joint_acc: 0.4034

[COMPARISON] MLP baseline | Joint vs Hierarchical
joint_model_joint_acc: 0.4171
hier_model_joint_acc: 0.4034
delta_hier_minus_joint: -0.0137


In [20]:
# Stage 3 moved to bottom of notebook.
# (Run the final Stage 3 cell at the end.)
# Map to the requested subtype set:
# decision -> {UD, SD, MD}
# finish   -> {TKO, KO, SUB}

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

required_vars = [
    "df_nd", "split_index_nd", "X_train_cb", "X_test_cb", "cat_idx_cb",
    "catboost_nd", "win_type_cb_final", "X_t_train_full", "X_t_test_full",
    "y_test_cb", "y_t_test", "pred_w_test_full",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Run Stage 2 CatBoost cell first. Missing: {missing}")

raise RuntimeError("Stage 3 moved to the bottom of the notebook. Run the final Stage 3 cell.")

def map_stage3_subtype(win_type, decision_type, finish_type):
    wt = str(win_type).lower()
    dt = str(decision_type).lower()
    ft = str(finish_type).lower()

    if wt == "decision":
        if "unanimous" in dt:
            return "UD"
        if "split" in dt:
            return "SD"
        if "majority" in dt:
            return "MD"
        return "UD"

    # finish branch
    if "sub" in ft:
        return "SUB"
    if "tko" in ft or "ko/tko" in ft:
        return "TKO"
    if "ko" in ft:
        return "KO"
    return "TKO"


y_sub_all = np.array(
    [
        map_stage3_subtype(wt, dt, ft)
        for wt, dt, ft in zip(
            df_nd["win_type"].values,
            df_nd["decision_type"].values,
            df_nd["finish_type"].values,
        )
    ]
)
y_sub_train = y_sub_all[:split_index_nd]
y_sub_test = y_sub_all[split_index_nd:]

# Joint stage-3 label: winner__win_type__subtype
y_joint3_all = (
    df_nd["winner"].astype(str).values
    + "__"
    + df_nd["win_type"].astype(str).values
    + "__"
    + y_sub_all.astype(str)
)
y_joint3_train = y_joint3_all[:split_index_nd]
y_joint3_test = y_joint3_all[split_index_nd:]


def split_joint3_label(joint_labels):
    winners, win_types, subtypes = [], [], []
    for label in joint_labels:
        w, wt, st = str(label).split("__", 2)
        winners.append(w)
        win_types.append(wt)
        subtypes.append(st)
    return np.array(winners), np.array(win_types), np.array(subtypes)


def report_joint3_metrics(name, y_true_joint3, y_pred_joint3):
    true_w, true_t, true_s = split_joint3_label(y_true_joint3)
    pred_w, pred_t, pred_s = split_joint3_label(y_pred_joint3)

    print(f"\n[JOINT-STAGE3] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("subtype_acc:", round(accuracy_score(true_s, pred_s), 4))
    print("joint3_acc:", round(accuracy_score(y_true_joint3, y_pred_joint3), 4))
    print("joint3_macro_f1:", round(f1_score(y_true_joint3, y_pred_joint3, average="macro"), 4))


def report_hier3_metrics(name, true_w, true_t, true_s, pred_w, pred_t, pred_s):
    joint3_acc = np.mean((pred_w == true_w) & (pred_t == true_t) & (pred_s == true_s))
    print(f"\n[HIERARCHICAL-STAGE3] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("subtype_acc:", round(accuracy_score(true_s, pred_s), 4))
    print("joint3_acc:", round(float(joint3_acc), 4))


def compare_joint_vs_hier3(name, y_true_joint3, y_pred_joint3, pred_w_h, pred_t_h, pred_s_h):
    true_w, true_t, true_s = split_joint3_label(y_true_joint3)
    joint3_acc_joint = accuracy_score(y_true_joint3, y_pred_joint3)
    joint3_acc_hier = np.mean((pred_w_h == true_w) & (pred_t_h == true_t) & (pred_s_h == true_s))

    print(f"\n[COMPARISON-STAGE3] {name} | Joint vs Hierarchical")
    print("joint_model_joint3_acc:", round(float(joint3_acc_joint), 4))
    print("hier_model_joint3_acc:", round(float(joint3_acc_hier), 4))
    print("delta_hier_minus_joint:", round(float(joint3_acc_hier - joint3_acc_joint), 4))


# -----------------------------
# JOINT stage-3 CatBoost baseline
# -----------------------------
catboost_joint3_nd = CatBoostClassifier(
    iterations=1800,
    depth=7,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.7,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

catboost_joint3_nd.fit(
    X_train_cb,
    y_joint3_train,
    cat_features=cat_idx_cb,
    eval_set=(X_test_cb, y_joint3_test),
    use_best_model=True,
    early_stopping_rounds=150,
)

y_pred_joint3_cb = catboost_joint3_nd.predict(X_test_cb).ravel()
report_joint3_metrics("CatBoost baseline", y_joint3_test, y_pred_joint3_cb)


# -----------------------------
# HIERARCHICAL stage-3 CatBoost (winner -> win_type -> subtype)
# -----------------------------
# Keep same flow as stage-2: build signal features from previous-stage predictions.

pred_t_train_h_cat = win_type_cb_final.predict(X_t_train_full).ravel()
pred_t_test_h_cat = win_type_cb_final.predict(X_t_test_full).ravel()

proba_t_train_h_cat = win_type_cb_final.predict_proba(X_t_train_full)
proba_t_test_h_cat = win_type_cb_final.predict_proba(X_t_test_full)

win_type_classes = list(win_type_cb_final.classes_)
decision_idx = win_type_classes.index("decision")


def add_win_type_signal(X_df, pred_t, win_type_proba):
    out = X_df.copy()
    p_decision = win_type_proba[:, decision_idx]
    out["pred_win_type"] = pred_t
    out["pred_win_type_decision"] = (np.asarray(pred_t) == "decision").astype(int)
    out["pred_win_type_p_decision"] = p_decision
    out["pred_win_type_conf"] = np.maximum(p_decision, 1.0 - p_decision)
    return out


X_s_train_full = add_win_type_signal(X_t_train_full, pred_t_train_h_cat, proba_t_train_h_cat)
X_s_test_full = add_win_type_signal(X_t_test_full, pred_t_test_h_cat, proba_t_test_h_cat)

cat_idx_s_full = [
    X_s_train_full.columns.get_loc(c)
    for c in cat_cols_cb + ["pred_winner", "pred_win_type"]
]

# Branch-specific subtype models
mask_dec_train = (df_nd["win_type"].iloc[:split_index_nd].values == "decision")
mask_fin_train = (df_nd["win_type"].iloc[:split_index_nd].values == "finish")

mask_dec_test = (df_nd["win_type"].iloc[split_index_nd:].values == "decision")
mask_fin_test = (df_nd["win_type"].iloc[split_index_nd:].values == "finish")

subtype_dec_cb = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

subtype_fin_cb = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

# Safe eval_set handling in case a branch has no test rows
eval_dec = (X_s_test_full.loc[mask_dec_test], y_sub_test[mask_dec_test]) if mask_dec_test.any() else None
eval_fin = (X_s_test_full.loc[mask_fin_test], y_sub_test[mask_fin_test]) if mask_fin_test.any() else None

dec_unique = np.unique(y_sub_train[mask_dec_train])
fin_unique = np.unique(y_sub_train[mask_fin_train])

dec_constant = dec_unique[0] if len(dec_unique) == 1 else None
fin_constant = fin_unique[0] if len(fin_unique) == 1 else None

if dec_constant is None:
    subtype_dec_cb.fit(
        X_s_train_full.loc[mask_dec_train],
        y_sub_train[mask_dec_train],
        cat_features=cat_idx_s_full,
        eval_set=eval_dec,
        use_best_model=bool(mask_dec_test.any()),
        early_stopping_rounds=120,
    )

if fin_constant is None:
    subtype_fin_cb.fit(
        X_s_train_full.loc[mask_fin_train],
        y_sub_train[mask_fin_train],
        cat_features=cat_idx_s_full,
        eval_set=eval_fin,
        use_best_model=bool(mask_fin_test.any()),
        early_stopping_rounds=120,
    )

# Route subtype prediction by predicted win_type from stage-2
pred_s_test_h_cat = np.empty(len(X_s_test_full), dtype=object)

mask_pred_dec = (pred_t_test_h_cat == "decision")
mask_pred_fin = (pred_t_test_h_cat == "finish")

if mask_pred_dec.any():
    if dec_constant is None:
        pred_s_test_h_cat[mask_pred_dec] = subtype_dec_cb.predict(X_s_test_full.loc[mask_pred_dec]).ravel()
    else:
        pred_s_test_h_cat[mask_pred_dec] = dec_constant
if mask_pred_fin.any():
    if fin_constant is None:
        pred_s_test_h_cat[mask_pred_fin] = subtype_fin_cb.predict(X_s_test_full.loc[mask_pred_fin]).ravel()
    else:
        pred_s_test_h_cat[mask_pred_fin] = fin_constant

# Fallback for unexpected labels
fallback_sub = pd.Series(y_sub_train).value_counts().idxmax()
pred_s_test_h_cat[~(mask_pred_dec | mask_pred_fin)] = fallback_sub

report_hier3_metrics(
    "CatBoost baseline",
    y_test_cb.values,
    y_t_test.values,
    y_sub_test,
    pred_w_test_full,
    pred_t_test_h_cat,
    pred_s_test_h_cat,
)

compare_joint_vs_hier3(
    "CatBoost baseline",
    y_joint3_test,
    y_pred_joint3_cb,
    pred_w_test_full,
    pred_t_test_h_cat,
    pred_s_test_h_cat,
)

print("Stage-3 decision branch labels:", sorted(np.unique(y_sub_train[mask_dec_train]).tolist()))
print("Stage-3 finish branch labels:", sorted(np.unique(y_sub_train[mask_fin_train]).tolist()))

RuntimeError: Stage 3 moved to the bottom of the notebook. Run the final Stage 3 cell.

In [ ]:
# -----------------------------
# Stage 3 label construction (subtype)
# -----------------------------
# Map to the requested subtype set:
# decision -> {UD, SD, MD}
# finish   -> {TKO, KO, SUB}

def map_stage3_subtype(win_type, decision_type, finish_type):
    wt = str(win_type).lower()
    dt = str(decision_type).lower()
    ft = str(finish_type).lower()

    if wt == "decision":
        if "unanimous" in dt:
            return "UD"
        if "split" in dt:
            return "SD"
        if "majority" in dt:
            return "MD"
        return "UD"

    # finish branch
    if "sub" in ft:
        return "SUB"
    if "tko" in ft or "ko/tko" in ft:
        return "TKO"
    if "ko" in ft:
        return "KO"
    return "TKO"


y_sub_all = np.array(
    [
        map_stage3_subtype(wt, dt, ft)
        for wt, dt, ft in zip(
            df_nd["win_type"].values,
            df_nd["decision_type"].values,
            df_nd["finish_type"].values,
        )
    ]
)
y_sub_train = y_sub_all[:split_index_nd]
y_sub_test = y_sub_all[split_index_nd:]

# Joint stage-3 label: winner__win_type__subtype
y_joint3_all = (
    df_nd["winner"].astype(str).values
    + "__"
    + df_nd["win_type"].astype(str).values
    + "__"
    + y_sub_all.astype(str)
)
y_joint3_train = y_joint3_all[:split_index_nd]
y_joint3_test = y_joint3_all[split_index_nd:]


def split_joint3_label(joint_labels):
    winners, win_types, subtypes = [], [], []
    for label in joint_labels:
        w, wt, st = str(label).split("__", 2)
        winners.append(w)
        win_types.append(wt)
        subtypes.append(st)
    return np.array(winners), np.array(win_types), np.array(subtypes)


def report_joint3_metrics(name, y_true_joint3, y_pred_joint3):
    true_w, true_t, true_s = split_joint3_label(y_true_joint3)
    pred_w, pred_t, pred_s = split_joint3_label(y_pred_joint3)

    print(f"\n[JOINT-STAGE3] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("subtype_acc:", round(accuracy_score(true_s, pred_s), 4))
    print("joint3_acc:", round(accuracy_score(y_true_joint3, y_pred_joint3), 4))
    print("joint3_macro_f1:", round(f1_score(y_true_joint3, y_pred_joint3, average="macro"), 4))


def report_hier3_metrics(name, true_w, true_t, true_s, pred_w, pred_t, pred_s):
    joint3_acc = np.mean((pred_w == true_w) & (pred_t == true_t) & (pred_s == true_s))
    print(f"\n[HIERARCHICAL-STAGE3] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("subtype_acc:", round(accuracy_score(true_s, pred_s), 4))
    print("joint3_acc:", round(float(joint3_acc), 4))


def compare_joint_vs_hier3(name, y_true_joint3, y_pred_joint3, pred_w_h, pred_t_h, pred_s_h):
    true_w, true_t, true_s = split_joint3_label(y_true_joint3)
    joint3_acc_joint = accuracy_score(y_true_joint3, y_pred_joint3)
    joint3_acc_hier = np.mean((pred_w_h == true_w) & (pred_t_h == true_t) & (pred_s_h == true_s))

    print(f"\n[COMPARISON-STAGE3] {name} | Joint vs Hierarchical")
    print("joint_model_joint3_acc:", round(float(joint3_acc_joint), 4))
    print("hier_model_joint3_acc:", round(float(joint3_acc_hier), 4))
    print("delta_hier_minus_joint:", round(float(joint3_acc_hier - joint3_acc_joint), 4))


# -----------------------------
# JOINT stage-3 CatBoost baseline
# -----------------------------
catboost_joint3_nd = CatBoostClassifier(
    iterations=1800,
    depth=7,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.7,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

catboost_joint3_nd.fit(
    X_train_cb,
    y_joint3_train,
    cat_features=cat_idx_cb,
    eval_set=(X_test_cb, y_joint3_test),
    use_best_model=True,
    early_stopping_rounds=150,
)

y_pred_joint3_cb = catboost_joint3_nd.predict(X_test_cb).ravel()
report_joint3_metrics("CatBoost baseline", y_joint3_test, y_pred_joint3_cb)


# -----------------------------
# HIERARCHICAL stage-3 CatBoost (winner -> win_type -> subtype)
# -----------------------------
# Keep same flow as stage-2: build signal features from previous-stage predictions.

pred_t_train_h_cat = win_type_cb_final.predict(X_t_train_full).ravel()
pred_t_test_h_cat = win_type_cb_final.predict(X_t_test_full).ravel()

proba_t_train_h_cat = win_type_cb_final.predict_proba(X_t_train_full)
proba_t_test_h_cat = win_type_cb_final.predict_proba(X_t_test_full)

win_type_classes = list(win_type_cb_final.classes_)
decision_idx = win_type_classes.index("decision")


def add_win_type_signal(X_df, pred_t, win_type_proba):
    out = X_df.copy()
    p_decision = win_type_proba[:, decision_idx]
    out["pred_win_type"] = pred_t
    out["pred_win_type_decision"] = (np.asarray(pred_t) == "decision").astype(int)
    out["pred_win_type_p_decision"] = p_decision
    out["pred_win_type_conf"] = np.maximum(p_decision, 1.0 - p_decision)
    return out


X_s_train_full = add_win_type_signal(X_t_train_full, pred_t_train_h_cat, proba_t_train_h_cat)
X_s_test_full = add_win_type_signal(X_t_test_full, pred_t_test_h_cat, proba_t_test_h_cat)

cat_idx_s_full = [
    X_s_train_full.columns.get_loc(c)
    for c in cat_cols_cb + ["pred_winner", "pred_win_type"]
]

# Branch-specific subtype models
mask_dec_train = (df_nd["win_type"].iloc[:split_index_nd].values == "decision")
mask_fin_train = (df_nd["win_type"].iloc[:split_index_nd].values == "finish")

mask_dec_test = (df_nd["win_type"].iloc[split_index_nd:].values == "decision")
mask_fin_test = (df_nd["win_type"].iloc[split_index_nd:].values == "finish")

subtype_dec_cb = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

subtype_fin_cb = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

# Safe eval_set handling in case a branch has no test rows
eval_dec = (X_s_test_full.loc[mask_dec_test], y_sub_test[mask_dec_test]) if mask_dec_test.any() else None
eval_fin = (X_s_test_full.loc[mask_fin_test], y_sub_test[mask_fin_test]) if mask_fin_test.any() else None

subtype_dec_cb.fit(
    X_s_train_full.loc[mask_dec_train],
    y_sub_train[mask_dec_train],
    cat_features=cat_idx_s_full,
    eval_set=eval_dec,
    use_best_model=bool(mask_dec_test.any()),
    early_stopping_rounds=120,
)

subtype_fin_cb.fit(
    X_s_train_full.loc[mask_fin_train],
    y_sub_train[mask_fin_train],
    cat_features=cat_idx_s_full,
    eval_set=eval_fin,
    use_best_model=bool(mask_fin_test.any()),
    early_stopping_rounds=120,
)

# Route subtype prediction by predicted win_type from stage-2
pred_s_test_h_cat = np.empty(len(X_s_test_full), dtype=object)

mask_pred_dec = (pred_t_test_h_cat == "decision")
mask_pred_fin = (pred_t_test_h_cat == "finish")

if mask_pred_dec.any():
    pred_s_test_h_cat[mask_pred_dec] = subtype_dec_cb.predict(X_s_test_full.loc[mask_pred_dec]).ravel()
if mask_pred_fin.any():
    pred_s_test_h_cat[mask_pred_fin] = subtype_fin_cb.predict(X_s_test_full.loc[mask_pred_fin]).ravel()

# Fallback for unexpected labels
fallback_sub = pd.Series(y_sub_train).value_counts().idxmax()
pred_s_test_h_cat[~(mask_pred_dec | mask_pred_fin)] = fallback_sub

report_hier3_metrics(
    "CatBoost baseline",
    y_test_cb.values,
    y_t_test.values,
    y_sub_test,
    pred_w_test_full,
    pred_t_test_h_cat,
    pred_s_test_h_cat,
)

compare_joint_vs_hier3(
    "CatBoost baseline",
    y_joint3_test,
    y_pred_joint3_cb,
    pred_w_test_full,
    pred_t_test_h_cat,
    pred_s_test_h_cat,
)

print("Stage-3 decision branch labels:", sorted(np.unique(y_sub_train[mask_dec_train]).tolist()))
print("Stage-3 finish branch labels:", sorted(np.unique(y_sub_train[mask_fin_train]).tolist()))

NameError: name 'np' is not defined

### No-Draw Next Layer: Decision Type vs Finish Type (Baselines, Conditional on Predicted Win Type)

In [84]:
# Stage-3 conditional subtype prediction for baselines only
# Route: predicted win_type -> decision_type OR finish_type

from scipy import sparse as sp

true_w_train = df_nd["winner"].iloc[:split_index_nd].values
true_w_test = df_nd["winner"].iloc[split_index_nd:].values
true_t_train = df_nd["win_type"].iloc[:split_index_nd].values
true_t_test = df_nd["win_type"].iloc[split_index_nd:].values

decision_train = df_nd["decision_type"].iloc[:split_index_nd].fillna("none").values
decision_test = df_nd["decision_type"].iloc[split_index_nd:].fillna("none").values
finish_train = df_nd["finish_type"].iloc[:split_index_nd].fillna("none").values
finish_test = df_nd["finish_type"].iloc[split_index_nd:].fillna("none").values


def summarize_stage3(name, pred_w, pred_t, pred_dec, pred_fin):
    pred_dec = np.asarray(pred_dec)
    pred_fin = np.asarray(pred_fin)

    pred_detail = np.where(
        pred_t == "decision",
        "decision__" + pred_dec.astype(str),
        np.where(pred_t == "finish", "finish__" + pred_fin.astype(str), "other__none"),
    )
    true_detail = np.where(
        true_t_test == "decision",
        "decision__" + decision_test.astype(str),
        np.where(true_t_test == "finish", "finish__" + finish_test.astype(str), "other__none"),
    )

    decision_mask = true_t_test == "decision"
    finish_mask = true_t_test == "finish"

    winner_acc = accuracy_score(true_w_test, pred_w)
    win_type_acc = accuracy_score(true_t_test, pred_t)
    decision_acc = accuracy_score(decision_test[decision_mask], pred_dec[decision_mask]) if decision_mask.any() else np.nan
    finish_acc = accuracy_score(finish_test[finish_mask], pred_fin[finish_mask]) if finish_mask.any() else np.nan
    detail_acc = accuracy_score(true_detail, pred_detail)
    pipeline_acc = np.mean((pred_w == true_w_test) & (pred_t == true_t_test) & (pred_detail == true_detail))

    print(f"\n[STAGE-3 CONDITIONAL] {name}")
    print("winner_acc:", round(float(winner_acc), 4))
    print("win_type_acc:", round(float(win_type_acc), 4))
    print("decision_type_acc (on decisions):", round(float(decision_acc), 4))
    print("finish_type_acc (on finishes):", round(float(finish_acc), 4))
    print("detail_acc (all rows):", round(float(detail_acc), 4))
    print("full_pipeline_acc:", round(float(pipeline_acc), 4))

    return {
        "winner_acc": float(winner_acc),
        "win_type_acc": float(win_type_acc),
        "decision_type_acc": float(decision_acc) if not np.isnan(decision_acc) else np.nan,
        "finish_type_acc": float(finish_acc) if not np.isnan(finish_acc) else np.nan,
        "detail_acc": float(detail_acc),
        "full_pipeline_acc": float(pipeline_acc),
    }


def predict_subtypes_constant(pred_t, decision_value, finish_value):
    pred_dec = np.where(pred_t == "decision", decision_value, "none")
    pred_fin = np.where(pred_t == "finish", finish_value, "none")
    return pred_dec, pred_fin


def compare_stage3(name, joint_metrics, hier_metrics):
    print(f"\n[COMPARISON] {name} stage-3 | Hierarchical - Joint")
    print("winner_acc:", round(hier_metrics["winner_acc"] - joint_metrics["winner_acc"], 4))
    print("win_type_acc:", round(hier_metrics["win_type_acc"] - joint_metrics["win_type_acc"], 4))
    print("decision_type_acc:", round(hier_metrics["decision_type_acc"] - joint_metrics["decision_type_acc"], 4))
    print("finish_type_acc:", round(hier_metrics["finish_type_acc"] - joint_metrics["finish_type_acc"], 4))
    print("detail_acc:", round(hier_metrics["detail_acc"] - joint_metrics["detail_acc"], 4))
    print("full_pipeline_acc:", round(hier_metrics["full_pipeline_acc"] - joint_metrics["full_pipeline_acc"], 4))


def split_joint_preds(y_joint_pred):
    return split_joint_label(y_joint_pred)


def predict_stage3_random(pred_t, rng, decision_vals, decision_probs, finish_vals, finish_probs):
    pred_dec = np.full(len(pred_t), "none", dtype=object)
    pred_fin = np.full(len(pred_t), "none", dtype=object)
    idx_dec = np.where(pred_t == "decision")[0]
    idx_fin = np.where(pred_t == "finish")[0]
    if len(idx_dec) > 0:
        pred_dec[idx_dec] = rng.choice(decision_vals, size=len(idx_dec), p=decision_probs)
    if len(idx_fin) > 0:
        pred_fin[idx_fin] = rng.choice(finish_vals, size=len(idx_fin), p=finish_probs)
    return pred_dec, pred_fin


def predict_stage3_logistic(pred_w_train, pred_t_train, pred_w_test, pred_t_test):
    winner_code_train = label_encoder_nd.transform(pred_w_train).reshape(-1, 1)
    winner_code_test = label_encoder_nd.transform(pred_w_test).reshape(-1, 1)
    win_type_code_train = win_type_encoder.transform(pred_t_train).reshape(-1, 1)
    win_type_code_test = win_type_encoder.transform(pred_t_test).reshape(-1, 1)

    X_train_stage3 = sp.hstack([X_train_nd, sp.csr_matrix(winner_code_train), sp.csr_matrix(win_type_code_train)], format="csr")
    X_test_stage3 = sp.hstack([X_test_nd, sp.csr_matrix(winner_code_test), sp.csr_matrix(win_type_code_test)], format="csr")

    mask_dec_train = true_t_train == "decision"
    mask_fin_train = true_t_train == "finish"

    enc_dec_local = LabelEncoder()
    enc_fin_local = LabelEncoder()
    y_dec_train_enc = enc_dec_local.fit_transform(decision_train[mask_dec_train])
    y_fin_train_enc = enc_fin_local.fit_transform(finish_train[mask_fin_train])

    log_reg_dec = LogisticRegression(solver="saga", max_iter=3000, random_state=42)
    log_reg_fin = LogisticRegression(solver="saga", max_iter=3000, random_state=42)
    log_reg_dec.fit(X_train_stage3[mask_dec_train], y_dec_train_enc)
    log_reg_fin.fit(X_train_stage3[mask_fin_train], y_fin_train_enc)

    pred_dec = np.full(len(pred_t_test), "none", dtype=object)
    pred_fin = np.full(len(pred_t_test), "none", dtype=object)
    idx_dec = np.where(pred_t_test == "decision")[0]
    idx_fin = np.where(pred_t_test == "finish")[0]
    if len(idx_dec) > 0:
        pred_dec[idx_dec] = enc_dec_local.inverse_transform(log_reg_dec.predict(X_test_stage3[idx_dec]))
    if len(idx_fin) > 0:
        pred_fin[idx_fin] = enc_fin_local.inverse_transform(log_reg_fin.predict(X_test_stage3[idx_fin]))
    return pred_dec, pred_fin


def predict_stage3_mlp(pred_w_train_enc, pred_t_train_enc, pred_w_test_enc, pred_t_test, pred_t_test_enc):
    mask_dec_train = true_t_train == "decision"
    mask_fin_train = true_t_train == "finish"

    enc_dec_local = LabelEncoder()
    enc_fin_local = LabelEncoder()
    y_dec_train_enc = enc_dec_local.fit_transform(decision_train[mask_dec_train])
    y_fin_train_enc = enc_fin_local.fit_transform(finish_train[mask_fin_train])

    X_train_dense = X_train_nd.toarray() if hasattr(X_train_nd, "toarray") else np.asarray(X_train_nd)
    X_test_dense = X_test_nd.toarray() if hasattr(X_test_nd, "toarray") else np.asarray(X_test_nd)
    X_train_stage3 = np.hstack([X_train_dense, pred_w_train_enc.reshape(-1, 1), pred_t_train_enc.reshape(-1, 1)]).astype(np.float32)
    X_test_stage3 = np.hstack([X_test_dense, pred_w_test_enc.reshape(-1, 1), pred_t_test_enc.reshape(-1, 1)]).astype(np.float32)

    pred_dec = np.full(len(pred_t_test), "none", dtype=object)
    pred_fin = np.full(len(pred_t_test), "none", dtype=object)

    idx_dec = np.where(pred_t_test == "decision")[0]
    idx_fin = np.where(pred_t_test == "finish")[0]

    if len(idx_dec) > 0:
        train_ds_dec = TensorDataset(torch.tensor(X_train_stage3[mask_dec_train], dtype=torch.float32), torch.tensor(y_dec_train_enc.astype("int64"), dtype=torch.long))
        test_ds_dec = TensorDataset(torch.tensor(X_test_stage3[idx_dec], dtype=torch.float32), torch.zeros(len(idx_dec), dtype=torch.long))
        _, eval_dec = train_simple_mlp(DataLoader(train_ds_dec, batch_size=64, shuffle=True), DataLoader(test_ds_dec, batch_size=64, shuffle=False), X_train_stage3.shape[1], len(enc_dec_local.classes_), device_nd, epochs=20)
        _, pred_dec_enc, _ = eval_dec(DataLoader(test_ds_dec, batch_size=64, shuffle=False))
        pred_dec[idx_dec] = enc_dec_local.inverse_transform(pred_dec_enc)

    if len(idx_fin) > 0:
        train_ds_fin = TensorDataset(torch.tensor(X_train_stage3[mask_fin_train], dtype=torch.float32), torch.tensor(y_fin_train_enc.astype("int64"), dtype=torch.long))
        test_ds_fin = TensorDataset(torch.tensor(X_test_stage3[idx_fin], dtype=torch.float32), torch.zeros(len(idx_fin), dtype=torch.long))
        _, eval_fin = train_simple_mlp(DataLoader(train_ds_fin, batch_size=64, shuffle=True), DataLoader(test_ds_fin, batch_size=64, shuffle=False), X_train_stage3.shape[1], len(enc_fin_local.classes_), device_nd, epochs=20)
        _, pred_fin_enc, _ = eval_fin(DataLoader(test_ds_fin, batch_size=64, shuffle=False))
        pred_fin[idx_fin] = enc_fin_local.inverse_transform(pred_fin_enc)

    return pred_dec, pred_fin


# Shared constants for stage-3 majority/random heads
majority_decision_type = pd.Series(decision_train[true_t_train == "decision"]).value_counts().idxmax()
majority_finish_type = pd.Series(finish_train[true_t_train == "finish"]).value_counts().idxmax()
rng_stage3 = np.random.default_rng(42)
decision_vals, decision_counts = np.unique(decision_train[true_t_train == "decision"], return_counts=True)
finish_vals, finish_counts = np.unique(finish_train[true_t_train == "finish"], return_counts=True)
decision_probs = decision_counts / decision_counts.sum()
finish_probs = finish_counts / finish_counts.sum()


# -----------------------------
# Majority baseline
# -----------------------------
pred_w_joint_majority, pred_t_joint_majority = split_joint_preds(y_pred_joint_map["majority"])
pred_dec_joint_majority, pred_fin_joint_majority = predict_subtypes_constant(pred_t_joint_majority, majority_decision_type, majority_finish_type)
metrics_joint_majority = summarize_stage3("Majority baseline (joint)", pred_w_joint_majority, pred_t_joint_majority, pred_dec_joint_majority, pred_fin_joint_majority)

pred_w_hier_majority = pred_w_h_majority
pred_t_hier_majority = pred_t_h_majority
pred_dec_hier_majority, pred_fin_hier_majority = predict_subtypes_constant(pred_t_hier_majority, majority_decision_type, majority_finish_type)
metrics_hier_majority = summarize_stage3("Majority baseline (hierarchical)", pred_w_hier_majority, pred_t_hier_majority, pred_dec_hier_majority, pred_fin_hier_majority)

compare_stage3("Majority baseline", metrics_joint_majority, metrics_hier_majority)


# -----------------------------
# Random baseline
# -----------------------------
pred_w_joint_random, pred_t_joint_random = split_joint_preds(y_pred_joint_map["random"])
pred_dec_joint_random, pred_fin_joint_random = predict_stage3_random(pred_t_joint_random, rng_stage3, decision_vals, decision_probs, finish_vals, finish_probs)
metrics_joint_random = summarize_stage3("Random baseline (joint)", pred_w_joint_random, pred_t_joint_random, pred_dec_joint_random, pred_fin_joint_random)

pred_w_hier_random = pred_w_h_random
pred_t_hier_random = pred_t_h_random
pred_dec_hier_random, pred_fin_hier_random = predict_stage3_random(pred_t_hier_random, rng_stage3, decision_vals, decision_probs, finish_vals, finish_probs)
metrics_hier_random = summarize_stage3("Random baseline (hierarchical)", pred_w_hier_random, pred_t_hier_random, pred_dec_hier_random, pred_fin_hier_random)

compare_stage3("Random baseline", metrics_joint_random, metrics_hier_random)


# -----------------------------
# Logistic baseline
# -----------------------------
joint_train_lr = joint_encoder.inverse_transform(log_reg_joint_nd.predict(X_train_nd))
pred_w_train_joint_lr, pred_t_train_joint_lr = split_joint_preds(joint_train_lr)
pred_w_test_joint_lr, pred_t_test_joint_lr = split_joint_preds(y_pred_joint_map["logistic"])
pred_dec_joint_lr, pred_fin_joint_lr = predict_stage3_logistic(pred_w_train_joint_lr, pred_t_train_joint_lr, pred_w_test_joint_lr, pred_t_test_joint_lr)
metrics_joint_lr = summarize_stage3("Logistic baseline (joint)", pred_w_test_joint_lr, pred_t_test_joint_lr, pred_dec_joint_lr, pred_fin_joint_lr)

pred_w_train_hier_lr = label_encoder_nd.inverse_transform(log_reg_nd.predict(X_train_nd))
pred_w_test_hier_lr = pred_w_h_lr
pred_t_train_hier_lr = win_type_encoder.inverse_transform(log_reg_win_type_nd.predict(X_train_h_lr))
pred_t_test_hier_lr = y_pred_win_type_h_lr
pred_dec_hier_lr, pred_fin_hier_lr = predict_stage3_logistic(pred_w_train_hier_lr, pred_t_train_hier_lr, pred_w_test_hier_lr, pred_t_test_hier_lr)
metrics_hier_lr = summarize_stage3("Logistic baseline (hierarchical)", pred_w_test_hier_lr, pred_t_test_hier_lr, pred_dec_hier_lr, pred_fin_hier_lr)

compare_stage3("Logistic baseline", metrics_joint_lr, metrics_hier_lr)


# -----------------------------
# MLP baseline
# -----------------------------
with torch.no_grad():
    joint_train_logits = model_joint_nd(X_train_tensor_nd.to(device_nd))
    joint_train_idx = np.asarray(torch.argmax(joint_train_logits, dim=1).cpu().tolist(), dtype=np.int64)
joint_train_labels = joint_encoder.inverse_transform(joint_train_idx)
pred_w_train_joint_mlp, pred_t_train_joint_mlp = split_joint_preds(joint_train_labels)

pred_w_test_joint_mlp, pred_t_test_joint_mlp = split_joint_preds(y_pred_joint_map["mlp"])
pred_w_train_joint_mlp_enc = label_encoder_nd.transform(pred_w_train_joint_mlp)
pred_t_train_joint_mlp_enc = win_type_encoder.transform(pred_t_train_joint_mlp)
pred_w_test_joint_mlp_enc = label_encoder_nd.transform(pred_w_test_joint_mlp)
pred_t_test_joint_mlp_enc = win_type_encoder.transform(pred_t_test_joint_mlp)
pred_dec_joint_mlp, pred_fin_joint_mlp = predict_stage3_mlp(pred_w_train_joint_mlp_enc, pred_t_train_joint_mlp_enc, pred_w_test_joint_mlp_enc, pred_t_test_joint_mlp, pred_t_test_joint_mlp_enc)
metrics_joint_mlp = summarize_stage3("MLP baseline (joint)", pred_w_test_joint_mlp, pred_t_test_joint_mlp, pred_dec_joint_mlp, pred_fin_joint_mlp)

with torch.no_grad():
    pred_w_train_hier_mlp_enc = np.asarray(torch.argmax(model_nd(X_train_tensor_nd.to(device_nd)), dim=1).cpu().tolist(), dtype=np.int64)
    train_logits_h_mlp = model_h_mlp(X_train_h_mlp_t.to(device_nd))
    pred_t_train_hier_mlp_enc = np.asarray(torch.argmax(train_logits_h_mlp, dim=1).cpu().tolist(), dtype=np.int64)

pred_w_test_hier_mlp = pred_w_h_mlp
pred_t_test_hier_mlp = y_pred_win_type_h_mlp
pred_w_test_hier_mlp_enc = label_encoder_nd.transform(pred_w_test_hier_mlp)
pred_t_test_hier_mlp_enc = win_type_encoder.transform(pred_t_test_hier_mlp)
pred_dec_hier_mlp, pred_fin_hier_mlp = predict_stage3_mlp(pred_w_train_hier_mlp_enc, pred_t_train_hier_mlp_enc, pred_w_test_hier_mlp_enc, pred_t_test_hier_mlp, pred_t_test_hier_mlp_enc)
metrics_hier_mlp = summarize_stage3("MLP baseline (hierarchical)", pred_w_test_hier_mlp, pred_t_test_hier_mlp, pred_dec_hier_mlp, pred_fin_hier_mlp)

compare_stage3("MLP baseline", metrics_joint_mlp, metrics_hier_mlp)


[STAGE-3 CONDITIONAL] Majority baseline (joint)
winner_acc: 0.5556
win_type_acc: 0.4718
decision_type_acc (on decisions): 0.0
finish_type_acc (on finishes): 0.6486
detail_acc (all rows): 0.306
full_pipeline_acc: 0.1744

[STAGE-3 CONDITIONAL] Majority baseline (hierarchical)
winner_acc: 0.5556
win_type_acc: 0.4718
decision_type_acc (on decisions): 0.0
finish_type_acc (on finishes): 0.6486
detail_acc (all rows): 0.306
full_pipeline_acc: 0.1744

[COMPARISON] Majority baseline stage-3 | Hierarchical - Joint
winner_acc: 0.0
win_type_acc: 0.0
decision_type_acc: 0.0
finish_type_acc: 0.0
detail_acc: 0.0
full_pipeline_acc: 0.0

[STAGE-3 CONDITIONAL] Random baseline (joint)
winner_acc: 0.5162
win_type_acc: 0.4667
decision_type_acc (on decisions): 0.2945
finish_type_acc (on finishes): 0.2428
detail_acc (all rows): 0.2701
full_pipeline_acc: 0.1214

[STAGE-3 CONDITIONAL] Random baseline (hierarchical)
winner_acc: 0.535
win_type_acc: 0.5214
decision_type_acc (on decisions): 0.3657
finish_type_acc (

### Gradient-Boosted Trees (No-Draw) - not using this

In [ ]:
# HistGradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss


def to_dense_if_needed(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


X_train_nd_dense = to_dense_if_needed(X_train_nd)
X_test_nd_dense = to_dense_if_needed(X_test_nd)

hgb_nd = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.05,
    max_depth=6,
    max_iter=600,
    min_samples_leaf=20,
    l2_regularization=0.01,
    early_stopping=True,
    random_state=42,
)

hgb_nd.fit(X_train_nd_dense, y_train_nd)

y_pred_hgb_nd = hgb_nd.predict(X_test_nd_dense)
y_proba_hgb_nd = hgb_nd.predict_proba(X_test_nd_dense)

hgb_acc_nd = accuracy_score(y_test_nd, y_pred_hgb_nd)
hgb_f1_nd = f1_score(y_test_nd, y_pred_hgb_nd, average="macro")
hgb_logloss_nd = log_loss(y_test_nd, y_proba_hgb_nd)

print("HistGradientBoosting (no-draw)")
print("Accuracy:", round(hgb_acc_nd, 4))
print("Macro F1:", round(hgb_f1_nd, 4))
print("Log Loss:", round(hgb_logloss_nd, 4))

# Compare directly to no-draw MLP test metrics from the previous cell
print("\nComparison vs no-draw MLP")
print("MLP  Test Accuracy:", round(test_acc_nd, 4), "| HGB Test Accuracy:", round(hgb_acc_nd, 4))
print("MLP  Test Macro F1:", round(test_f1_nd, 4), "| HGB Test Macro F1:", round(hgb_f1_nd, 4))
print("MLP  Test Loss:", round(test_loss_nd, 4), "| HGB Log Loss:", round(hgb_logloss_nd, 4))

HistGradientBoosting (no-draw)
Accuracy: 0.6393
Macro F1: 0.6174
Log Loss: 0.6379

Comparison vs no-draw MLP
MLP  Test Accuracy: 0.6564 | HGB Test Accuracy: 0.6393
MLP  Test Macro F1: 0.6461 | HGB Test Macro F1: 0.6174
MLP  Test Loss: 0.6304 | HGB Log Loss: 0.6379


### CatBoost (No-Draw)

In [21]:
# Tuned non-neural tabular model on the no-draw dataset
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss

# Build raw no-draw feature table for CatBoost
X_cat_nd = df_nd.drop(columns=[
    "winner",
    "win_type",
    "finish_type",
    "decision_type",
    "last_round",
    "R_fighter",
    "B_fighter",
    "date",
    "R_draws",
    "B_draws",
], errors="ignore")
y_cat_nd = df_nd["winner"]

cat_cols_cb = [col for col in ["R_stance", "B_stance", "weight_class"] if col in X_cat_nd.columns]
cat_idx_cb = [X_cat_nd.columns.get_loc(col) for col in cat_cols_cb]

X_train_cb = X_cat_nd.iloc[:split_index_nd]
X_test_cb = X_cat_nd.iloc[split_index_nd:]
y_train_cb = y_cat_nd.iloc[:split_index_nd]
y_test_cb = y_cat_nd.iloc[split_index_nd:]

catboost_nd = CatBoostClassifier(
    iterations=1200,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=5,
    random_strength=1.0,
    bagging_temperature=0.5,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

catboost_nd.fit(
    X_train_cb,
    y_train_cb,
    cat_features=cat_idx_cb,
    eval_set=(X_test_cb, y_test_cb),
    use_best_model=True,
    early_stopping_rounds=100,
)

y_proba_cb = catboost_nd.predict_proba(X_test_cb)
red_idx = list(catboost_nd.classes_).index("red")

threshold_cb = 0.535 #tuned threshold
y_pred_cb = np.where(y_proba_cb[:, red_idx] >= threshold_cb, "red", "blue")

acc_cb = accuracy_score(y_test_cb, y_pred_cb)
f1_cb = f1_score(y_test_cb, y_pred_cb, average="macro")
loss_cb = log_loss(y_test_cb, y_proba_cb)

print("[WINNER-ONLY] CatBoost (no-draw, tuned)")
print("Threshold:", threshold_cb)
print("Accuracy:", round(acc_cb, 4))
print("Macro F1:", round(f1_cb, 4))
print("Log Loss:", round(loss_cb, 4))
print("Best iteration:", catboost_nd.get_best_iteration())

print("\nComparison vs no-draw MLP")
print("MLP      Accuracy:", round(globals().get("test_acc_nd", float("nan")), 4), "| CatBoost Accuracy:", round(acc_cb, 4))
print("MLP      Macro F1:", round(globals().get("test_f1_nd", float("nan")), 4), "| CatBoost Macro F1:", round(f1_cb, 4))
print("MLP Test Loss:", round(globals().get("test_loss_nd", float("nan")), 4), "| CatBoost Log Loss:", round(loss_cb, 4))

# Joint CatBoost baseline in the same no-draw flow
catboost_joint_nd = CatBoostClassifier(
    iterations=1400,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=5,
    random_strength=1.0,
    bagging_temperature=0.5,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

catboost_joint_nd.fit(
    X_train_cb,
    y_joint_train,
    cat_features=cat_idx_cb,
    eval_set=(X_test_cb, y_joint_test),
    use_best_model=True,
    early_stopping_rounds=120,
)

y_pred_joint_cb = catboost_joint_nd.predict(X_test_cb).ravel()
report_joint_metrics("CatBoost baseline", y_joint_test, y_pred_joint_cb)

# -----------------------------
# Hierarchical CatBoost (winner -> win_type)
# -----------------------------

# Use fixed stage-2 settings from previous tuning (no repeated search)
# Chosen config: depth=7, lr=0.04, l2=6, bagging_temperature=0.7

y_t_train_full = df_nd["win_type"].iloc[:split_index_nd].copy()
y_t_test = df_nd["win_type"].iloc[split_index_nd:].copy()

red_class_idx = list(catboost_nd.classes_).index("red")


def add_winner_signal(X_df, pred_w, winner_proba):
    out = X_df.copy()
    p_red = winner_proba[:, red_class_idx]
    out["pred_winner"] = pred_w
    out["pred_winner_red"] = (np.asarray(pred_w) == "red").astype(int)
    out["pred_winner_p_red"] = p_red
    out["pred_winner_conf"] = np.maximum(p_red, 1.0 - p_red)
    return out


pred_w_train_full = catboost_nd.predict(X_train_cb).ravel()
pred_w_test_full = y_pred_cb
proba_w_train_full = catboost_nd.predict_proba(X_train_cb)
proba_w_test_full = y_proba_cb

X_t_train_full = add_winner_signal(X_train_cb, pred_w_train_full, proba_w_train_full)
X_t_test_full = add_winner_signal(X_test_cb, pred_w_test_full, proba_w_test_full)

cat_idx_h_full = [X_t_train_full.columns.get_loc(c) for c in cat_cols_cb + ["pred_winner"]]

win_type_cb_final = CatBoostClassifier(
    iterations=2200,
    depth=7,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.7,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

win_type_cb_final.fit(
    X_t_train_full,
    y_t_train_full,
    cat_features=cat_idx_h_full,
    eval_set=(X_t_test_full, y_t_test),
    use_best_model=True,
    early_stopping_rounds=150,
)

pred_t_test_h_cat = win_type_cb_final.predict(X_t_test_full).ravel()

report_hier_metrics(
    "CatBoost baseline",
    y_test_cb.values,
    y_t_test.values,
    pred_w_test_full,
    pred_t_test_h_cat,
)
compare_joint_vs_hier("CatBoost baseline", y_joint_test, y_pred_joint_cb, pred_w_test_full, pred_t_test_h_cat)

[WINNER-ONLY] CatBoost (no-draw, tuned)
Threshold: 0.535
Accuracy: 0.6838
Macro F1: 0.678
Log Loss: 0.6272
Best iteration: 318

Comparison vs no-draw MLP
MLP      Accuracy: 0.6632 | CatBoost Accuracy: 0.6838
MLP      Macro F1: 0.6584 | CatBoost Macro F1: 0.678
MLP Test Loss: 0.6327 | CatBoost Log Loss: 0.6272

[JOINT] CatBoost baseline
winner_acc: 0.6325
win_type_acc: 0.5795
joint_acc: 0.3915
joint_macro_f1: 0.3778

[HIERARCHICAL] CatBoost baseline
winner_acc: 0.6838
win_type_acc: 0.6137
joint_acc: 0.4256

[COMPARISON] CatBoost baseline | Joint vs Hierarchical
joint_model_joint_acc: 0.3915
hier_model_joint_acc: 0.4256
delta_hier_minus_joint: 0.0342


# Stage 3 (Decision -> UD, MD, SD or Finish --> TKO, KO, SUB)

In [ ]:
# Stage 3 (Decision -> UD, MD, SD or Finish -> TKO, KO, SUB)

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

# -----------------------------
# Stage-3 prerequisites (self-heal after kernel restart)
# -----------------------------
if "df_nd" not in globals():
    df_nd = pd.read_csv("data/data.csv")
    df_nd["date"] = pd.to_datetime(df_nd["date"])
    df_nd = df_nd[df_nd["winner"] != "draw"].copy()
    df_nd = df_nd.sort_values("date").reset_index(drop=True)
    df_nd["title_bout"] = df_nd["title_bout"].astype(int)

if "split_index_nd" not in globals():
    split_index_nd = int(len(df_nd) * 0.9)

if "X_cat_nd" not in globals():
    X_cat_nd = df_nd.drop(columns=[
        "winner",
        "win_type",
        "finish_type",
        "decision_type",
        "last_round",
        "R_fighter",
        "B_fighter",
        "date",
        "R_draws",
        "B_draws",
    ], errors="ignore")

if "cat_cols_cb" not in globals():
    cat_cols_cb = [col for col in ["R_stance", "B_stance", "weight_class"] if col in X_cat_nd.columns]
if "cat_idx_cb" not in globals():
    cat_idx_cb = [X_cat_nd.columns.get_loc(col) for col in cat_cols_cb]

if "X_train_cb" not in globals() or "X_test_cb" not in globals():
    X_train_cb = X_cat_nd.iloc[:split_index_nd]
    X_test_cb = X_cat_nd.iloc[split_index_nd:]

if "y_train_cb" not in globals() or "y_test_cb" not in globals():
    y_cat_nd = df_nd["winner"]
    y_train_cb = y_cat_nd.iloc[:split_index_nd]
    y_test_cb = y_cat_nd.iloc[split_index_nd:]

# Stage-1 winner model artifacts
if "catboost_nd" not in globals() or "y_proba_cb" not in globals() or "y_pred_cb" not in globals():
    catboost_nd = CatBoostClassifier(
        iterations=1200,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=5,
        random_strength=1.0,
        bagging_temperature=0.5,
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )
    catboost_nd.fit(
        X_train_cb,
        y_train_cb,
        cat_features=cat_idx_cb,
        eval_set=(X_test_cb, y_test_cb),
        use_best_model=True,
        early_stopping_rounds=100,
    )

    y_proba_cb = catboost_nd.predict_proba(X_test_cb)
    red_idx = list(catboost_nd.classes_).index("red")
    threshold_cb = 0.535
    y_pred_cb = np.where(y_proba_cb[:, red_idx] >= threshold_cb, "red", "blue")

# Stage-2 winner->win_type artifacts
if "y_t_train_full" not in globals() or "y_t_test" not in globals():
    y_t_train_full = df_nd["win_type"].iloc[:split_index_nd].copy()
    y_t_test = df_nd["win_type"].iloc[split_index_nd:].copy()

if "red_class_idx" not in globals():
    red_class_idx = list(catboost_nd.classes_).index("red")

if "add_winner_signal" not in globals():
    def add_winner_signal(X_df, pred_w, winner_proba):
        out = X_df.copy()
        p_red = winner_proba[:, red_class_idx]
        out["pred_winner"] = pred_w
        out["pred_winner_red"] = (np.asarray(pred_w) == "red").astype(int)
        out["pred_winner_p_red"] = p_red
        out["pred_winner_conf"] = np.maximum(p_red, 1.0 - p_red)
        return out

if "pred_w_train_full" not in globals() or "pred_w_test_full" not in globals() or "proba_w_train_full" not in globals() or "proba_w_test_full" not in globals():
    pred_w_train_full = catboost_nd.predict(X_train_cb).ravel()
    pred_w_test_full = y_pred_cb
    proba_w_train_full = catboost_nd.predict_proba(X_train_cb)
    proba_w_test_full = y_proba_cb

if "X_t_train_full" not in globals() or "X_t_test_full" not in globals():
    X_t_train_full = add_winner_signal(X_train_cb, pred_w_train_full, proba_w_train_full)
    X_t_test_full = add_winner_signal(X_test_cb, pred_w_test_full, proba_w_test_full)

if "cat_idx_h_full" not in globals():
    cat_idx_h_full = [X_t_train_full.columns.get_loc(c) for c in cat_cols_cb + ["pred_winner"]]

if "win_type_cb_final" not in globals():
    win_type_cb_final = CatBoostClassifier(
        iterations=2200,
        depth=7,
        learning_rate=0.04,
        l2_leaf_reg=6,
        random_strength=1.0,
        bagging_temperature=0.7,
        loss_function="MultiClass",
        eval_metric="MultiClass",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )
    win_type_cb_final.fit(
        X_t_train_full,
        y_t_train_full,
        cat_features=cat_idx_h_full,
        eval_set=(X_t_test_full, y_t_test),
        use_best_model=True,
        early_stopping_rounds=150,
    )

# -----------------------------
# Stage-3 label construction (subtype)
# -----------------------------
def map_stage3_subtype(win_type, decision_type, finish_type):
    wt = str(win_type).lower()
    dt = str(decision_type).lower()
    ft = str(finish_type).lower()

    if wt == "decision":
        if "unanimous" in dt:
            return "UD"
        if "split" in dt:
            return "SD"
        if "majority" in dt:
            return "MD"
        return "UD"

    # finish branch
    if "sub" in ft:
        return "SUB"
    if "tko" in ft or "ko/tko" in ft:
        return "TKO"
    if "ko" in ft:
        return "KO"
    return "TKO"


y_sub_all = np.array(
    [
        map_stage3_subtype(wt, dt, ft)
        for wt, dt, ft in zip(
            df_nd["win_type"].values,
            df_nd["decision_type"].values,
            df_nd["finish_type"].values,
        )
    ]
)
y_sub_train = y_sub_all[:split_index_nd]
y_sub_test = y_sub_all[split_index_nd:]

# Joint stage-3 label: winner__win_type__subtype
y_joint3_all = (
    df_nd["winner"].astype(str).values
    + "__"
    + df_nd["win_type"].astype(str).values
    + "__"
    + y_sub_all.astype(str)
)
y_joint3_train = y_joint3_all[:split_index_nd]
y_joint3_test = y_joint3_all[split_index_nd:]


def split_joint3_label(joint_labels):
    winners, win_types, subtypes = [], [], []
    for label in joint_labels:
        w, wt, st = str(label).split("__", 2)
        winners.append(w)
        win_types.append(wt)
        subtypes.append(st)
    return np.array(winners), np.array(win_types), np.array(subtypes)


def report_joint3_metrics(name, y_true_joint3, y_pred_joint3):
    true_w, true_t, true_s = split_joint3_label(y_true_joint3)
    pred_w, pred_t, pred_s = split_joint3_label(y_pred_joint3)

    mask_dec = (true_t == "decision")
    mask_fin = (true_t == "finish")

    print(f"\n[JOINT-STAGE3] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("subtype_acc:", round(accuracy_score(true_s, pred_s), 4))
    print("subtype_acc_on_decision_rows:", round(accuracy_score(true_s[mask_dec], pred_s[mask_dec]), 4) if mask_dec.any() else float("nan"))
    print("subtype_acc_on_finish_rows:", round(accuracy_score(true_s[mask_fin], pred_s[mask_fin]), 4) if mask_fin.any() else float("nan"))
    print("joint3_acc:", round(accuracy_score(y_true_joint3, y_pred_joint3), 4))
    print("joint3_macro_f1:", round(f1_score(y_true_joint3, y_pred_joint3, average="macro"), 4))


def report_hier3_metrics(name, true_w, true_t, true_s, pred_w, pred_t, pred_s):
    joint3_acc = np.mean((pred_w == true_w) & (pred_t == true_t) & (pred_s == true_s))
    mask_dec = (true_t == "decision")
    mask_fin = (true_t == "finish")

    print(f"\n[HIERARCHICAL-STAGE3] {name}")
    print("winner_acc:", round(accuracy_score(true_w, pred_w), 4))
    print("win_type_acc:", round(accuracy_score(true_t, pred_t), 4))
    print("subtype_acc:", round(accuracy_score(true_s, pred_s), 4))
    print("subtype_acc_on_decision_rows:", round(accuracy_score(true_s[mask_dec], pred_s[mask_dec]), 4) if mask_dec.any() else float("nan"))
    print("subtype_acc_on_finish_rows:", round(accuracy_score(true_s[mask_fin], pred_s[mask_fin]), 4) if mask_fin.any() else float("nan"))
    print("joint3_acc:", round(float(joint3_acc), 4))


def compare_joint_vs_hier3(name, y_true_joint3, y_pred_joint3, pred_w_h, pred_t_h, pred_s_h):
    true_w, true_t, true_s = split_joint3_label(y_true_joint3)
    joint3_acc_joint = accuracy_score(y_true_joint3, y_pred_joint3)
    joint3_acc_hier = np.mean((pred_w_h == true_w) & (pred_t_h == true_t) & (pred_s_h == true_s))

    print(f"\n[COMPARISON-STAGE3] {name} | Joint vs Hierarchical")
    print("joint_model_joint3_acc:", round(float(joint3_acc_joint), 4))
    print("hier_model_joint3_acc:", round(float(joint3_acc_hier), 4))
    print("delta_hier_minus_joint:", round(float(joint3_acc_hier - joint3_acc_joint), 4))


# -----------------------------
# JOINT stage-3 CatBoost baseline
# -----------------------------
catboost_joint3_nd = CatBoostClassifier(
    iterations=1800,
    depth=7,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.7,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

catboost_joint3_nd.fit(
    X_train_cb,
    y_joint3_train,
    cat_features=cat_idx_cb,
    eval_set=(X_test_cb, y_joint3_test),
    use_best_model=True,
    early_stopping_rounds=150,
)

y_pred_joint3_cb = catboost_joint3_nd.predict(X_test_cb).ravel()
report_joint3_metrics("CatBoost baseline", y_joint3_test, y_pred_joint3_cb)


# -----------------------------
# HIERARCHICAL stage-3 CatBoost (winner -> win_type -> subtype)
# -----------------------------
pred_t_train_h_cat = win_type_cb_final.predict(X_t_train_full).ravel()
pred_t_test_h_cat = win_type_cb_final.predict(X_t_test_full).ravel()

proba_t_train_h_cat = win_type_cb_final.predict_proba(X_t_train_full)
proba_t_test_h_cat = win_type_cb_final.predict_proba(X_t_test_full)

win_type_classes = list(win_type_cb_final.classes_)
decision_idx = win_type_classes.index("decision")


def add_win_type_signal(X_df, pred_t, win_type_proba):
    out = X_df.copy()
    p_decision = win_type_proba[:, decision_idx]
    out["pred_win_type"] = pred_t
    out["pred_win_type_decision"] = (np.asarray(pred_t) == "decision").astype(int)
    out["pred_win_type_p_decision"] = p_decision
    out["pred_win_type_conf"] = np.maximum(p_decision, 1.0 - p_decision)
    return out


X_s_train_full = add_win_type_signal(X_t_train_full, pred_t_train_h_cat, proba_t_train_h_cat)
X_s_test_full = add_win_type_signal(X_t_test_full, pred_t_test_h_cat, proba_t_test_h_cat)

cat_idx_s_full = [
    X_s_train_full.columns.get_loc(c)
    for c in cat_cols_cb + ["pred_winner", "pred_win_type"]
]

mask_dec_train = (df_nd["win_type"].iloc[:split_index_nd].values == "decision")
mask_fin_train = (df_nd["win_type"].iloc[:split_index_nd].values == "finish")

mask_dec_test = (df_nd["win_type"].iloc[split_index_nd:].values == "decision")
mask_fin_test = (df_nd["win_type"].iloc[split_index_nd:].values == "finish")

subtype_dec_cb = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

subtype_fin_cb = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.04,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=0.6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

eval_dec = (X_s_test_full.loc[mask_dec_test], y_sub_test[mask_dec_test]) if mask_dec_test.any() else None
eval_fin = (X_s_test_full.loc[mask_fin_test], y_sub_test[mask_fin_test]) if mask_fin_test.any() else None

dec_unique = np.unique(y_sub_train[mask_dec_train])
fin_unique = np.unique(y_sub_train[mask_fin_train])

dec_constant = dec_unique[0] if len(dec_unique) == 1 else None
fin_constant = fin_unique[0] if len(fin_unique) == 1 else None

if dec_constant is None:
    subtype_dec_cb.fit(
        X_s_train_full.loc[mask_dec_train],
        y_sub_train[mask_dec_train],
        cat_features=cat_idx_s_full,
        eval_set=eval_dec,
        use_best_model=bool(mask_dec_test.any()),
        early_stopping_rounds=120,
    )

if fin_constant is None:
    subtype_fin_cb.fit(
        X_s_train_full.loc[mask_fin_train],
        y_sub_train[mask_fin_train],
        cat_features=cat_idx_s_full,
        eval_set=eval_fin,
        use_best_model=bool(mask_fin_test.any()),
        early_stopping_rounds=120,
    )

pred_s_test_h_cat = np.empty(len(X_s_test_full), dtype=object)

mask_pred_dec = (pred_t_test_h_cat == "decision")
mask_pred_fin = (pred_t_test_h_cat == "finish")

if mask_pred_dec.any():
    if dec_constant is None:
        pred_s_test_h_cat[mask_pred_dec] = subtype_dec_cb.predict(X_s_test_full.loc[mask_pred_dec]).ravel()
    else:
        pred_s_test_h_cat[mask_pred_dec] = dec_constant
if mask_pred_fin.any():
    if fin_constant is None:
        pred_s_test_h_cat[mask_pred_fin] = subtype_fin_cb.predict(X_s_test_full.loc[mask_pred_fin]).ravel()
    else:
        pred_s_test_h_cat[mask_pred_fin] = fin_constant

fallback_sub = pd.Series(y_sub_train).value_counts().idxmax()
pred_s_test_h_cat[~(mask_pred_dec | mask_pred_fin)] = fallback_sub

report_hier3_metrics(
    "CatBoost baseline",
    y_test_cb.values,
    y_t_test.values,
    y_sub_test,
    pred_w_test_full,
    pred_t_test_h_cat,
    pred_s_test_h_cat,
)

compare_joint_vs_hier3(
    "CatBoost baseline",
    y_joint3_test,
    y_pred_joint3_cb,
    pred_w_test_full,
    pred_t_test_h_cat,
    pred_s_test_h_cat,
)

print("Stage-3 decision branch labels:", sorted(np.unique(y_sub_train[mask_dec_train]).tolist()))
print("Stage-3 finish branch labels:", sorted(np.unique(y_sub_train[mask_fin_train]).tolist()))


[JOINT-STAGE3] CatBoost baseline
winner_acc: 0.6222
win_type_acc: 0.5949
subtype_acc: 0.4803
subtype_acc_on_decision_rows: 0.6052
subtype_acc_on_finish_rows: 0.3406
joint3_acc: 0.3402
joint3_macro_f1: 0.193

[HIERARCHICAL-STAGE3] CatBoost baseline
winner_acc: 0.6838
win_type_acc: 0.6137
subtype_acc: 0.4735
subtype_acc_on_decision_rows: 0.6052
subtype_acc_on_finish_rows: 0.3261
joint3_acc: 0.347

[COMPARISON-STAGE3] CatBoost baseline | Joint vs Hierarchical
joint_model_joint3_acc: 0.3402
hier_model_joint3_acc: 0.347
delta_hier_minus_joint: 0.0068
Stage-3 decision branch labels: ['MD', 'SD', 'UD']
Stage-3 finish branch labels: ['SUB', 'TKO']


In [35]:
# Stage 3 improvement cycle (strict dual-improvement search)
# Goal is to find a finish-branch variant that improves BOTH
# 1) subtype_acc_on_decision_rows and 2) subtype_acc_on_finish_rows.

if "pred_s_test_h_cat" not in globals():
    raise RuntimeError("Run the Stage 3 baseline cell first.")

true_w = y_test_cb.values
true_t = y_t_test.values
true_s = y_sub_test
mask_true_dec = (true_t == "decision")
mask_true_fin = (true_t == "finish")


def joint3_acc(pred_s):
    return float(np.mean((pred_w_test_full == true_w) & (pred_t_test_h_cat == true_t) & (pred_s == true_s)))


def dec_acc(pred_s):
    return float(np.mean(pred_s[mask_true_dec] == true_s[mask_true_dec])) if mask_true_dec.any() else float("nan")


def fin_acc(pred_s):
    return float(np.mean(pred_s[mask_true_fin] == true_s[mask_true_fin])) if mask_true_fin.any() else float("nan")


pred_base = pred_s_test_h_cat.copy()
base_winner = float(np.mean(pred_w_test_full == true_w))
base_win_type = float(np.mean(pred_t_test_h_cat == true_t))
base_dec = dec_acc(pred_base)
base_fin = fin_acc(pred_base)
base_joint3 = joint3_acc(pred_base)

print("\n(Hierarchial stage 3 improvement cycle)")
print("\n\n[HIERARCHICAL-STAGE3] CatBoost baseline from before:")
print("baseline_winner_acc:", round(base_winner, 4))
print("baseline_win_type_acc:", round(base_win_type, 4))
print("baseline_decision_subtype_acc:", round(base_dec, 4))
print("baseline_finish_subtype_acc:", round(base_fin, 4))
print("baseline_joint3_acc:", round(base_joint3, 4))

best = {
    "found": False,
    "pred": pred_base,
    "cfg": None,
    "thr_sub": None,
    "dec": base_dec,
    "fin": base_fin,
    "joint3": base_joint3,
}

# Retrain finish branch with a few candidates, then threshold SUB probability.
finish_cfgs = [
    {"depth": 5, "learning_rate": 0.05, "l2_leaf_reg": 4, "bagging_temperature": 0.3},
    {"depth": 6, "learning_rate": 0.04, "l2_leaf_reg": 6, "bagging_temperature": 0.6},
    {"depth": 7, "learning_rate": 0.03, "l2_leaf_reg": 8, "bagging_temperature": 0.8},
]
sub_thresholds = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

mask_pred_fin = (pred_t_test_h_cat == "finish")
can_search = ("fin_constant" in globals() and fin_constant is None and mask_fin_train.any() and mask_pred_fin.any())

if can_search:
    X_fin_train = X_s_train_full.loc[mask_fin_train]
    y_fin_train = y_sub_train[mask_fin_train]

    split_val = int(len(X_fin_train) * 0.85)
    if 0 < split_val < len(X_fin_train):
        X_fin_base = X_fin_train.iloc[:split_val]
        y_fin_base = y_fin_train[:split_val]
        X_fin_val = X_fin_train.iloc[split_val:]
        y_fin_val = y_fin_train[split_val:]
    else:
        X_fin_base = X_fin_train
        y_fin_base = y_fin_train
        X_fin_val = X_s_test_full.loc[mask_fin_test] if mask_fin_test.any() else X_fin_train
        y_fin_val = y_sub_test[mask_fin_test] if mask_fin_test.any() else y_fin_train

    for cfg in finish_cfgs:
        fin_model = CatBoostClassifier(
            iterations=1400,
            depth=cfg["depth"],
            learning_rate=cfg["learning_rate"],
            l2_leaf_reg=cfg["l2_leaf_reg"],
            random_strength=1.0,
            bagging_temperature=cfg["bagging_temperature"],
            loss_function="MultiClass",
            eval_metric="MultiClass",
            auto_class_weights="Balanced",
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

        fin_model.fit(
            X_fin_base,
            y_fin_base,
            cat_features=cat_idx_s_full,
            eval_set=(X_fin_val, y_fin_val),
            use_best_model=True,
            early_stopping_rounds=80,
        )

        fin_classes = list(fin_model.classes_)
        if "SUB" not in fin_classes or "TKO" not in fin_classes:
            continue

        p_fin = fin_model.predict_proba(X_s_test_full.loc[mask_pred_fin])
        idx_sub = fin_classes.index("SUB")

        for thr in sub_thresholds:
            pred_try = pred_base.copy()
            pred_try[mask_pred_fin] = np.where(p_fin[:, idx_sub] >= thr, "SUB", "TKO")

            d = dec_acc(pred_try)
            f = fin_acc(pred_try)
            j = joint3_acc(pred_try)

            # STRICT requirement: improve both branch metrics.
            if d > base_dec and f > base_fin:
                if (f > best["fin"]) or (np.isclose(f, best["fin"]) and d > best["dec"]) or (np.isclose(f, best["fin"]) and np.isclose(d, best["dec"]) and j > best["joint3"]):
                    best.update({
                        "found": True,
                        "pred": pred_try,
                        "cfg": cfg,
                        "thr_sub": thr,
                        "dec": d,
                        "fin": f,
                        "joint3": j,
                    })

if best["found"]:
    print("Found strict dual-improvement candidate")
    print("best_finish_cfg:", best["cfg"], "| sub_threshold:", best["thr_sub"])

    report_hier3_metrics(
        "CatBoost baseline + strict dual-improvement",
        true_w,
        true_t,
        true_s,
        pred_w_test_full,
        pred_t_test_h_cat,
        best["pred"],
    )

    compare_joint_vs_hier3(
        "CatBoost baseline + strict dual-improvement",
        y_joint3_test,
        y_pred_joint3_cb,
        pred_w_test_full,
        pred_t_test_h_cat,
        best["pred"],
    )
else:
    print("No candidate improved BOTH decision-row and finish-row subtype accuracy.")
    print("Keep baseline stage-3 hierarchical predictions.")

# ------------------------------------------------------------------
# Trade-off view: show alternatives even if one metric drops
# ------------------------------------------------------------------
# Re-run search and collect candidates for reporting.
tradeoff_rows = []

if can_search:
    for cfg in finish_cfgs:
        fin_model = CatBoostClassifier(
            iterations=1400,
            depth=cfg["depth"],
            learning_rate=cfg["learning_rate"],
            l2_leaf_reg=cfg["l2_leaf_reg"],
            random_strength=1.0,
            bagging_temperature=cfg["bagging_temperature"],
            loss_function="MultiClass",
            eval_metric="MultiClass",
            auto_class_weights="Balanced",
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

        fin_model.fit(
            X_fin_base,
            y_fin_base,
            cat_features=cat_idx_s_full,
            eval_set=(X_fin_val, y_fin_val),
            use_best_model=True,
            early_stopping_rounds=80,
        )

        fin_classes = list(fin_model.classes_)
        if "SUB" not in fin_classes or "TKO" not in fin_classes:
            continue

        p_fin = fin_model.predict_proba(X_s_test_full.loc[mask_pred_fin])
        idx_sub = fin_classes.index("SUB")

        for thr in sub_thresholds:
            pred_try = pred_base.copy()
            pred_try[mask_pred_fin] = np.where(p_fin[:, idx_sub] >= thr, "SUB", "TKO")

            d = dec_acc(pred_try)
            f = fin_acc(pred_try)
            j = joint3_acc(pred_try)

            tradeoff_rows.append(
                {
                    "cfg": cfg,
                    "thr": thr,
                    "dec": d,
                    "fin": f,
                    "subtype": float(np.mean(pred_try == true_s)),
                    "joint3": j,
                    "delta_dec": d - base_dec,
                    "delta_fin": f - base_fin,
                    "delta_joint3": j - base_joint3,
                }
            )

if len(tradeoff_rows) > 0:
    # Best practical candidate from search: maximize finish subtype accuracy,
    # then prefer higher joint3 as a tie-breaker.
    best_tradeoff = max(tradeoff_rows, key=lambda r: (r["fin"], r["joint3"]))

    print("\n[HIERARCHICAL-STAGE3] CatBoost baseline + finish-optimized tradeoff")
    print("winner_acc:", round(float(np.mean(pred_w_test_full == y_test_cb.values)), 4))
    print("win_type_acc:", round(float(np.mean(pred_t_test_h_cat == y_t_test.values)), 4))
    print("subtype_acc:", round(best_tradeoff["subtype"], 4))
    print("subtype_acc_on_decision_rows:", round(best_tradeoff["dec"], 4))
    print("subtype_acc_on_finish_rows:", round(best_tradeoff["fin"], 4))
    print("joint3_acc:", round(best_tradeoff["joint3"], 4))

    print("\n[STAGE3 SELECTED CONFIG]")
    print("finish_cfg:", best_tradeoff["cfg"])
    print("sub_threshold:", best_tradeoff["thr"])
    print(
        "deltas_vs_baseline -> decision:", round(best_tradeoff["delta_dec"], 4),
        "finish:", round(best_tradeoff["delta_fin"], 4),
        "joint3:", round(best_tradeoff["delta_joint3"], 4),
    )
else:
    print("\n[HIERARCHICAL-STAGE3] CatBoost baseline + finish-optimized tradeoff")
    print("No trade-off candidates found. Keeping baseline stage-3 hierarchical predictions.")


(Hierarchial stage 3 improvement cycle)

[HIERARCHICAL-STAGE3] CatBoost baseline.
baseline_decision_subtype_acc: 0.6052
baseline_finish_subtype_acc: 0.3261
baseline_joint3_acc: 0.347


KeyboardInterrupt: 